# 01 · Build the frequency tables (M2)

Builds `data/assets/ko_freq.json.gz` and `data/assets/zh_freq.json.gz`, the speech-vs-formal counts behind the contrast detector.

- **Runtime: CPU is enough.** No GPU needed for this notebook (Runtime → Change runtime type → CPU).
- **Time:** roughly 30–60 minutes, mostly downloading and segmenting.
- **Disk:** everything happens in Colab's own storage. Only a small zip (a few MB) comes back to your laptop.

**Before you start:** set `REPO` in the first cell. If the repo is **private**, add a GitHub token as a Colab secret named `GH_TOKEN` (key icon in the left sidebar → add `GH_TOKEN` → turn on notebook access). A fine-grained token with read-only *Contents* access to this one repo is enough.

Then run all cells (Runtime → Run all).

In [ ]:
# ---- settings ----
REPO = "https://github.com/YOUR_GITHUB_USERNAME/genAI-final.git"   # <- change this
BRANCH = "main"
SPEECH_LINES = 1_000_000   # subtitle lines per language
FORMAL_LINES = 500_000     # Wikipedia sentences per language

In [ ]:
# ---- get the code ----
import os, subprocess
token = None
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    pass  # public repo, or secret not set
url = REPO.replace("https://", f"https://{token}@") if token else REPO
if os.path.exists("/content/genAI-final/.git"):
    subprocess.run(["git", "-C", "/content/genAI-final", "pull", "-q"], check=True)
else:
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/genAI-final"], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit("Clone failed. Check REPO, and for a private repo the GH_TOKEN secret.\n"
                         + r.stderr.replace(token or "\0", "***"))
os.chdir("/content/genAI-final")
!git log --oneline -1

In [ ]:
# ---- install build dependencies (Colab already has most of them) ----
!pip install -q -r requirements-build.txt

## Korean
Subtitles come from OPUS OpenSubtitles; the script finds the right file through the OPUS API and streams only the first `SPEECH_LINES` lines. Wikipedia is streamed from HuggingFace and split into sentences.

In [ ]:
!python scripts/fetch_corpora.py subtitles --lang ko --max-lines {SPEECH_LINES}
!python scripts/fetch_corpora.py wiki --lang ko --max-lines {FORMAL_LINES}
!head -n 5 corpora/ko_speech.txt; echo ...; head -n 3 corpora/ko_formal.txt

In [ ]:
!python scripts/build_assets.py freq --lang ko --speech corpora/ko_speech.txt --formal corpora/ko_formal.txt --max-lines {SPEECH_LINES}

## Mandarin
Both sides are converted to Simplified characters with OpenCC, so the contrast does not mistake Traditional vs Simplified script for speech vs writing.

In [ ]:
!python scripts/fetch_corpora.py subtitles --lang zh --max-lines {SPEECH_LINES}
!python scripts/fetch_corpora.py wiki --lang zh --max-lines {FORMAL_LINES}
!head -n 5 corpora/zh_speech.txt; echo ...; head -n 3 corpora/zh_formal.txt

In [ ]:
!python scripts/build_assets.py freq --lang zh --speech corpora/zh_speech.txt --formal corpora/zh_formal.txt --max-lines {SPEECH_LINES}

## Check the tables
Each report shows the score distribution, where known slang ranks against everyday words, and what the detector would flag on the demo transcripts at thresholds 1, 2 and 3. **Read the "Known items vs everyday words" line:** the contrast signal is only useful if lexicon items score clearly higher.

In [ ]:
!python scripts/inspect_freq.py --lang ko

In [ ]:
!python scripts/inspect_freq.py --lang zh

## Download
Downloads `freq_tables.zip` (a few MB). Unzip it **into your repo folder**, so the files land in `data/assets/` and `reports/`, then commit `data/assets/*_freq.json.gz` and `reports/`.

In [ ]:
!ls -lh data/assets/*_freq.json.gz
!zip -q -r freq_tables.zip data/assets/ko_freq.json.gz data/assets/zh_freq.json.gz reports/
from google.colab import files
files.download("freq_tables.zip")